In [25]:
import os

import torch
from torch import Tensor

import torch_geometric
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from tqdm import tqdm, trange

In [2]:
# CONSTANTS 
DOMAIN_SIZE = 1000
DATA_FOLDER = os.path.join('data', 'boids')
RAW_DATA_FOLDER = os.path.join(DATA_FOLDER, 'raw', '')
PROCESSED_DATA_FOLDER = os.path.join(DATA_FOLDER, 'processed', '')

In [3]:
def square_torus_distance(p1, p2, width=1.0, height=1.0):
    """
    Compute the squared torus distance between two points p1 and p2 in a box of given width and height.

    Args:
        p1: Tensor of shape (num_points, 2) representing the first point(s)
        p2: Tensor of shape (num_points, 2) representing the second point(s)
        width: Width of the box
        height: Height of the box
    Returns:
        dist: Tensor of shape (num_points,) representing the squared torus distance between p1 and p2
    """
    dx = p1[:, 0] - p2[:, 0]
    dy = p1[:, 1] - p2[:, 1]
    dx = dx - width * torch.round(dx / width)
    dy = dy - height * torch.round(dy / height)
    return dx ** 2 + dy ** 2

def pbc_direction(p1: Tensor, p2: Tensor) -> Tensor:
    """
    Compute the direction from p1 to p2 considering periodic boundary conditions in a unit square.
    Args:
        p1: Tensor of shape (num_points, 2) representing the first point(s)
        p2: Tensor of shape (num_points, 2) representing the second point(s)
    Returns:
        direction: Tensor of shape (num_points, 2) representing the direction from p1 to p2 considering PBCs
    """
    return p2 - p1 - torch.round(p2 - p1)


In [4]:
class AR_Boids_Dataset(InMemoryDataset):
    def __init__(self, raw_data_path, processed_data_path, root=None, transform=None, pre_transform=None, post_transform=None, solution_idx_range=(0, 25), timesteps=1000, processed_file_name="AR3_Boids.pt"):
        self.raw_data_path = raw_data_path
        self.processed_data_path = processed_data_path
        self.solution_idx_range = solution_idx_range
        self.timesteps = timesteps
        self.processed_file_name = processed_file_name
        self.pre_transform = pre_transform
        self.transform = transform
        self.post_transform = post_transform
        super(AR_Boids_Dataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def processed_file_names(self):
        return [self.processed_file_name]

    @property
    def raw_file_names(self):
        return [pfn for pfn in os.listdir(self.raw_data_path) if (self.solution_idx_range[0] <= int(pfn.split("_")[-1][:-4]) < self.solution_idx_range[1])]
    
    def download(self):
        pass
    
    def __len__(self):
        return (self.timesteps - 1) * (self.solution_idx_range[1] - self.solution_idx_range[0])

    def process(self):
        positions_list = []
        data_list = []
        for idx, raw_path in enumerate(self.raw_file_names):
            trajectory = np.load(self.raw_data_path + raw_path)

            if self.transform is not None:
                trajectory = self.transform(trajectory)
                
            # Add the initial positions to the positions list
            positions_list.append(trajectory[0, :, :2])

            for t in trange(trajectory.shape[0] - 1):
                x = torch.tensor(trajectory[t], dtype=torch.float)
                y = torch.tensor(trajectory[t+1], dtype=torch.float)
                
                # Create fully connected graph
                n = trajectory.shape[1]
                edge_index = torch.tensor([[i, j] for i in range(n) for j in range(n) if i != j], dtype=torch.long).t().contiguous()
                
                data = Data(x=x, y=y, edge_index=edge_index)
                if self.post_transform is not None:
                    data = self.post_transform(data)
                
                data_list.append(data)
                
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_data_path+self.processed_file_name)
        torch.save(torch.tensor(positions_list), self.processed_data_path+"positions_"+self.processed_file_name)

    def __getitem__(self, idx):
        return self.get(idx)
    
    def __repr__(self):
        return f'{self.__class__.__name__}({len(self)})'

In [5]:
train_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(0, 15), 
    timesteps=1000, 
    processed_file_name="AR3_Boids_Equivariant.pt",
    transform=lambda traj: traj / DOMAIN_SIZE
)

validation_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(16, 25), 
    timesteps=1000, 
    processed_file_name="AR3_VAL_Boids_Equivariant.pt",
    transform=lambda traj: traj / DOMAIN_SIZE
)

Processing...
  0%|          | 0/999 [00:00<?, ?it/s]

100%|██████████| 999/999 [00:00<00:00, 3294.24it/s]
C:\Users\20212202\AppData\Local\Temp\ipykernel_27808\2626778711.py:56: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  torch.save(torch.tensor(positions_list), self.processed_data_path+"positions_"+self.processed_file_name)
Done!
Processing...
100%|██████████| 999/999 [00:00<00:00, 3328.54it/s]
Done!


In [146]:
class GeometricFlowMatchingModel(torch.nn.Module):
    def __init__(self):
        super(GeometricFlowMatchingModel, self).__init__()

    def forward(self, t: Tensor, data_t: Data, data_c: Data) -> Tensor:
        raise NotImplementedError("This method should be implemented in a subclass")

    def generate(self, data_c: Data, data_0: Data, n_euler_steps: int, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).to(data_0.x.device)
        data_t = data_0.clone()

        for i in range(n_euler_steps):
            data_t.x += (time_steps[i+1] - time_steps[i]) * self(t=time_steps[i].unsqueeze(-1), data_t=data_t, data_c=data_c)
            data_t.x[:, :2] %= 1.0

        return data_t.x

In [147]:
class EGNNLayer(torch.nn.Module):
    def __init__(self, h_dim=16, m_dim=16, hidden_dim=16):
        super(EGNNLayer, self).__init__()

        # Constants
        self.edge_dim = 3
        self.width = 1.0
        self.height = 1.0

        # edge messages
        self.phi_e = torch.nn.Sequential(
            torch.nn.Linear(h_dim * 2 + self.edge_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, m_dim),
            torch.nn.SiLU()
        )

        # speed
        self.phi_v = torch.nn.Sequential(
            torch.nn.Linear(m_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, 1)
        )

        # force
        phi_x_last_layer = torch.nn.Linear(hidden_dim, 1, bias=False)
        torch.nn.init.xavier_uniform_(phi_x_last_layer.weight, gain=0.001)
        self.phi_x = torch.nn.Sequential(
            torch.nn.Linear(m_dim, hidden_dim),
            torch.nn.SiLU(),
            phi_x_last_layer
        )

        # new hidden state
        self.node_embedding_nn = torch.nn.Sequential(
            torch.nn.Linear(h_dim + m_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, h_dim)
        )
    
    def phi_h(self, h, m):
        agg = torch.cat([h, m], dim=1)
        out = self.node_embedding_nn(agg)
        return out + h

    def calculate_edge_attributes(self, x, edge_index):
        edge_attr = torch.zeros((edge_index.shape[1], self.edge_dim), dtype=torch.float, device=x.device)

        # Square torus distance (to get distance between boids with PBCs)
        edge_attr[:, 0] = square_torus_distance(x[edge_index[0], :2], x[edge_index[1], :2], width=self.width, height=self.height)

        # Cosine similarity of velocities (to get alignment of velocities)
        edge_attr[:, 1] = torch.nn.functional.cosine_similarity(x[edge_index[0], 2:], x[edge_index[1], 2:], dim=1)

        # Calculate l2 norm between velocities (to get speed difference)
        edge_attr[:, 2] = (torch.norm(x[edge_index[1], 2:], dim=1) - torch.norm(x[edge_index[0], 2:], dim=1)).pow(2)

        return edge_attr

    def forward(self, x, edge_index, h, v_init):
        # Calculate edge features
        mij = self.phi_e(torch.cat([h[edge_index[0]], h[edge_index[1]], self.calculate_edge_attributes(x, edge_index).detach()], dim=1))

        # Add repulsion/attraction
        x[:, 2:] = self.phi_v(h) * v_init + scatter(
            pbc_direction(x[edge_index[0], :2], x[edge_index[1], :2]) * self.phi_x(mij), 
            edge_index[0], 
            0, 
            dim_size=x.shape[0], 
            reduce='sum'
        )

        # Update position
        x[:, :2] += x[:, 2:]
        x[:, 0] %= self.width
        x[:, 1] %= self.height
        
        # Update h
        h = self.phi_h(h, scatter(mij, edge_index[0], 0, dim_size=h.size(0), reduce='sum'))

        return h, x
    
class FlowEGNNModel(GeometricFlowMatchingModel):
    def __init__(self, h_dim=16, m_dim=16, hidden_dim=16, layers=1):
        super(FlowEGNNModel, self).__init__()
        self.h_dim = h_dim
        self.time_encoding = torch.nn.Linear(1, h_dim)
        self.embedding = EGNNLayer(h_dim, m_dim, hidden_dim)
        self.layers = torch.nn.ModuleList([EGNNLayer(h_dim, m_dim, hidden_dim) for _ in range(layers)])

    def forward(self, t: Tensor, data_t: Data, data_c: Data):
        x, edge_index = data_t.x.clone().detach(), data_t.edge_index
        x_init = x.clone()
        # v_init_direction = torch.nn.functional.normalize(x[:, 2:], dim=1)

        h = self.time_encoding(t.expand(x.shape[:-1]).unsqueeze(-1))
        h, _ = self.embedding(data_c.x.clone(), data_c.edge_index, h, data_c.x[:, 2:].clone())

        for mp_layer in self.layers:
            h, x = mp_layer(x, edge_index, h, x_init[:, 2:])

        # Create output
        out = torch.zeros_like(x)
        out[:, :2] = pbc_direction(x_init[:, :2], x[:, :2])
        out[:, 2:] = x[:, 2:] - x_init[:, 2:]

        return out


In [249]:
def boids_path_sampler(t, x_1, x_c, sigma):
    # Initialize x_t and x_dot_t
    x_t = torch.zeros_like(x_1)
    x_dot_t = torch.zeros_like(x_1)
    
    # Sample standard normal noise
    noise = sigma * torch.randn_like(x_1)

    # Compute x_t and x_dot_t
    x_t[:, :2] = x_c[:, :2] + pbc_direction(x_c[:, :2], x_1[:, :2]) * t + (1 - t) * noise[:, :2]
    x_t[:, 2:] = x_c[:, 2:] + (x_1[:, 2:] - x_c[:, 2:]) * t + (1 - t) * noise[:, 2:]
    x_dot_t[:, :2] = pbc_direction(x_c[:, :2], x_1[:, :2]) - noise[:, :2]
    x_dot_t[:, 2:] = (x_1[:, 2:] - x_c[:, 2:]) - noise[:, 2:]
    
    return x_t, x_dot_t

def ICFM_Boids_training(vf: GeometricFlowMatchingModel, dataset: AR_Boids_Dataset, n_epochs: int, sigma: float, device='cuda'):
    optimizer = torch.optim.Adam(vf.parameters(), 1e-2)
    loss_fn = torch.nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        for data in tqdm(dataset, leave=False):
            data_c = data.clone().to(device)
            data_t = data.clone().to(device)

            x_c = data_t.x
            x_1 = data_t.y
            t = torch.rand(1).to(device)

            x_t, u = boids_path_sampler(t, x_1, x_c, sigma)

            # noise = torch.randn_like(x_0)
            # x_t = (1 - t) * x_0 + t * x_1 + (1 - t) * sigma * noise
            
            # # u = x_1 - x_0
            # u = torch.zeros_like(x_0)
            # u[:, :2] = pbc_direction(x_0[:, :2], x_1[:, :2])
            # u[:, 2:] = x_1[:, 2:] - x_0[:, 2:]

            data_t.x = x_t
            
            optimizer.zero_grad()
            loss = loss_fn(vf(t=t, data_t=data_t, data_c=data_c), u)
            loss.backward()
            optimizer.step()
        print('epoch: ', epoch, ', loss: ', loss.item())
        loss_hist.append(loss.item())    

    return loss_hist

In [250]:
flow_model = FlowEGNNModel(layers=1).to('cuda')
sigma = 0.001
loss_hist_ICFM = ICFM_Boids_training(flow_model, train_dataset, n_epochs=2, sigma=sigma)


  0%|          | 0/14985 [00:00<?, ?it/s]

epoch:  0 , loss:  2.1446267055580392e-06


epoch:  1 , loss:  9.205006108459202e-07


In [232]:
def keep_01(data):
    return data[0:2, :, :] / DOMAIN_SIZE

initial_states_validation_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(16, 25), 
    timesteps=2, 
    processed_file_name="AR1_VAL_init.pt",
    transform=keep_01
)

In [233]:
# # Quick test 
# trajectory = np.load(os.path.join(RAW_DATA_FOLDER, 'boids_trajectory_25.npy'))
# initial_data = torch.tensor(trajectory[0], dtype=torch.float32) / DOMAIN_SIZE
# initial_position = initial_data[:, :2]
# initial_data = Data(x=initial_data[:, 2:], edge_index=torch.tensor([[i, j] for i in range(initial_data.shape[0]) for j in range(initial_data.shape[0]) if i != j], dtype=torch.long).t().contiguous(), edge_attr=torch.tensor([[pbc_distance(initial_position[i, 0], initial_position[j, 0], initial_position[i, 1], initial_position[j, 1])] for i in range(initial_data.shape[0]) for j in range(initial_data.shape[0]) if i != j], dtype=torch.float).t().contiguous())

# print(initial_position.shape)


In [251]:
@torch.no_grad()
def boids_rollout(model: GeometricFlowMatchingModel, init_data: Data, timesteps=1000, device='cuda', mode="residual", sigma=0.001):
    """
    Predict the rollouts of the model on the dataset starting from the initial state

    Args:
        model: PyTorch model
        dataset: PyTorch dataset
        timesteps: Number of timesteps to predict
        device: Device to run the model on
        mode: "residual" or "direct"
        - In the solution above, we used the "residual" mode, where the model predicts the change in position and velocity
        - In the "direct" mode, the model predicts the position and velocity directly (if you do not intend to use this mode, you can ignore this argument)
        width: Width of the PBC box
        height: Height of the PBC box
    Returns:
        rollouts: Rollouts of the model on the dataset
        - Should be a torch tensor of shape (Batch, Timesteps, Boids, Node_dim)
    """
    # Allocate memory for the rollouts
    rollouts = torch.empty((timesteps, *init_data.x.shape), device=device)

    # Get the initial state
    data_0 = init_data.clone().to(device)
    data_c = init_data.clone().to(device)
    current_position = init_data.x.clone().to(device)

    for t in trange(timesteps):
        data_0.x += torch.randn_like(data_0.x) * sigma
        out_data = model.generate(data_c=data_c, data_0=data_0, n_euler_steps=10)

        # Update
        if mode == "direct":
            current_position = out_data
        elif mode == "residual":
            current_position += out_data
        
        # Wrap around the positions
        current_position[:, 0] = current_position[:, 0] % 1.0
        current_position[:, 1] = current_position[:, 1] % 1.0

        # Update the model data
        data_0.x = current_position
        data_c.x = current_position

        # Store the current state
        rollouts[t] = data_c.x.clone().cpu()

    return rollouts


In [252]:
flow_model.eval()
data_sample = initial_states_validation_dataset[0].clone().to('cuda')
rollout = boids_rollout(flow_model, data_sample, timesteps=100, device='cuda', mode="direct", sigma=sigma)

print(rollout.shape)
print(rollout[-1])

100%|██████████| 100/100 [00:06<00:00, 15.65it/s]

torch.Size([100, 25, 4])
tensor([[ 1.2246e-02,  8.1868e-01,  2.2397e-03,  8.2583e-04],
        [ 2.5934e-01,  7.2247e-01, -3.3819e-03, -1.4286e-03],
        [ 6.8754e-01,  8.1799e-01, -1.3446e-03, -6.0424e-03],
        [ 7.7184e-01,  4.3069e-01, -2.0253e-03, -1.9752e-03],
        [ 1.2290e-01,  5.1603e-01, -1.0297e-03,  3.1866e-03],
        [ 9.7422e-01,  7.8877e-01,  3.2205e-04, -1.6363e-03],
        [ 1.4680e-01,  7.3064e-01,  1.0526e-03,  6.0231e-03],
        [ 2.8209e-01,  6.7593e-01, -8.2941e-04,  5.3723e-03],
        [ 3.3494e-01,  6.6703e-01,  5.1044e-03,  9.0272e-05],
        [ 6.2453e-01,  2.6067e-01, -6.9440e-03,  4.7467e-03],
        [ 2.1965e-01,  4.8587e-01, -9.3889e-03, -2.9468e-04],
        [ 3.2802e-02,  5.8625e-01,  9.5527e-03, -3.3712e-03],
        [ 6.0938e-01,  9.6317e-02,  6.7397e-03,  3.4866e-04],
        [ 7.6494e-01,  5.3521e-01,  1.9610e-03, -3.0038e-03],
        [ 5.4023e-01,  8.7690e-01,  2.3788e-03, -1.3454e-03],
        [ 7.1964e-01,  7.9341e-01,  1.2063e-0

In [246]:
# flow_model.generate(data_sample, data_sample, n_euler_steps=10)
# flow_model(0.0 * torch.ones(1).to('cuda'), data_sample, data_sample)
# rollout[0]
# data_sample.x

In [247]:
def animate_rollout(rollouts, output_path="output/rollout.gif", width = 1.0, height = 1.0, max_timesteps=100):
    # rollouts of shape (Timesteps, Boids, Node_dim)
    
    # Create output directory if it does not exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Parse rollouts
    timesteps, num_boids, node_dim = rollouts.shape
    rollouts = rollouts.cpu().numpy()

    # Initialize the figure and axis
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1)

    quiv = ax.quiver(rollouts[0, :, 0], rollouts[0, :, 1], rollouts[0, :, 2], rollouts[0, :, 3])
    scat = ax.scatter(rollouts[0, :, 0], rollouts[0, :, 1])
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_aspect('equal', adjustable='box')

    def update(frame):
        quiv.set_offsets(rollouts[frame, :, :2])
        quiv.set_UVC(rollouts[frame, :, 2], rollouts[frame, :, 3])
        scat.set_offsets(rollouts[frame, :, :2])
        return scat, quiv

    ani = animation.FuncAnimation(fig, update, frames=min(timesteps, max_timesteps), blit=False, interval=150)
    ani.save(output_path, writer="ffmpeg")
    plt.close()

In [ ]:
animate_rollout(rollout, output_path="output/test.gif")